### Objective

This notebook is the EMA cross GHBot sweep harness.

It clones `notebooks/emacross.ipynb`, keeps the same signal/data/runtime
shape, and swaps the notebook-local executor from Tradebot to GHBot.

The GHBot config shape follows:

- `D:/rust/nuutrader6/workspace/hcbots/templates/btc-ghbot-simnet.json5`
- EMA sweep timing follows the existing notebook harness.


In [ ]:
# Notebook Setup
# Keep nbconvert and VS Code launches on the repo root.

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

# Sweep Control
# User-edited sweep template plus notebook-only controls.

template = """
[sweep]
mode = "fast"
start_bot_id = 1

[params]
fast = [10]
slow = [50]

[botrun.runtime]
bot_id = 1
mode = "sweep"
max_loop = 0
loop_seconds = 1.0

[botrun.market]
symbol = "BTCUSDT"
interval = "1m"

[botrun.backtest]
start = "2025-01-01"
stop = "2025-03-31T23:59:59"
data_dir = "workspace/data"

[[botrun.signalers]]
name = "emacross"
interval = "1m"

[botrun.signalers.params]
fast = 9
slow = 21

[botrun.executor]
name = "ghbot"
upper_bound_pct = "5"
lower_bound_pct = "5"
max_cycles = 0

[botrun.executor.grid]
grid_enabled = true
grid_account = "sgrid"
grid_direction = "long"
grid_investment_usdc = "2000"
grid_leverage = 3
grid_levels = 41
grid_min_order_age_s = 60
grid_slippage_reserve_pct = "1"
grid_spread_multiplier = "1"
grid_winactive_order = 39
grid_win_interval_s = 30
grid_win_active_recalc_levels = 3
grid_level_reentry_cooldown_s = 5
grid_close_on_stop = true

[botrun.executor.hedge]
hedge_enabled = true
hedge_account = "shedge"
hedge_direction = "short"
hedge_investment_usdc = "2000"
hedge_leverage = 3
hedge_reserve_pct = "1"
hedge_entry_pct = "1"
hedge_sl_pct = "0.5"
hedge_take_profit_pct = "1"
hedge_trailing_stop_pct = "1"
hedge_cooldown_s = 600
hedge_grace_period_s = 5
hedge_max_retries = 3
hedge_sl_peg_mode = "entry_price"

[botrun.executor.risk]
risk_max_dd_pct = "10"
risk_max_hedge_losses = 5

[botrun.executor.audit]
audit_window_secs = 3600

[botrun.risk]
score = 1
"""

# 0 creates a new sweep. Nonzero loads and reruns that sweep_id.
SWEEP_ID = 0
FEE_PCT = 0.0


In [ ]:
# Imports
# All notebook imports live here.

from dataclasses import dataclass, field
from datetime import datetime, timezone
import itertools
import json
from pathlib import Path
import time
import tomllib

from IPython.display import HTML, IFrame, display
from sqlalchemy import select

from nuubot import Nuubot
from nuubot.core.dtypes import Bar, MarketSnapshot, Signal
from nuubot.core.market_data import date_ms, load_binance_bars
from nuubot.core.models.mconfig import BotrunConfig, SweepConfig
from nuubot.datastore import SweepRow, SweeprunRow


In [ ]:
# Prepare SweepRun
# Parse template, validate it, create or load sweep records, then load all sweeprun records.

def scalar_json(config: BotrunConfig) -> str:
    return json.dumps(config.model_dump(mode="json"), sort_keys=True, separators=(",", ":"))


def apply_param(botrun: dict, key: str, value) -> None:
    signaler_params = botrun["signalers"][0].setdefault("params", {})
    if key in signaler_params:
        signaler_params[key] = value
        return
    if key in botrun["executor"]:
        botrun["executor"][key] = value
        return
    raise ValueError(f"sweep param has no target in botrun: {key}")


def expand_botruns(config: SweepConfig) -> list[BotrunConfig]:
    params = config.params or {}
    keys = list(params)
    values = [value if isinstance(value, list) else [value] for value in params.values()]
    expanded = []
    for combo in itertools.product(*values) if values else [()]:
        botrun = config.botrun.model_dump(mode="json")
        for key, value in zip(keys, combo, strict=True):
            apply_param(botrun, key, value)
        expanded.append(BotrunConfig.model_validate(botrun))
    return expanded


def create_sweep_record(nuubot, config: SweepConfig, manifest_json: str) -> int:
    botruns = expand_botruns(config)
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        sweep = SweepRow(
            sweep_desc="emacross_ghbot_notebook_smoke",
            config_json=manifest_json,
            results_json="{}",
            status="configured",
            sweeprun_count=len(botruns),
        )
        session.add(sweep)
        session.flush()
        for index, botrun in enumerate(botruns):
            session.add(SweeprunRow(
                sweep_id=sweep.sweep_id,
                sweeprun_index=index,
                config_json=scalar_json(botrun),
                results_json="{}",
                status="configured",
            ))
        session.commit()
        return sweep.sweep_id


def load_sweep_records(nuubot, sweep_id: int) -> tuple[SweepRow, list[SweeprunRow]]:
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        sweep = session.get(SweepRow, sweep_id)
        if sweep is None:
            raise ValueError(f"sweep not found: {sweep_id}")
        sweepruns = list(session.scalars(select(SweeprunRow).where(SweeprunRow.sweep_id == sweep_id).order_by(SweeprunRow.sweeprun_index)))
        if not sweepruns:
            raise ValueError(f"sweepruns not found for sweep: {sweep_id}")
        session.expunge(sweep)
        for row in sweepruns:
            session.expunge(row)
        return sweep, sweepruns


nuubot = Nuubot().setup()
try:
    if SWEEP_ID == 0:
        template_data = tomllib.loads(template)
        template_data["botrun"]["backtest"]["data_dir"] = f"{nuubot.config.paths.data_dir}/binance/raw/spot/monthly/klines"
        template_config = SweepConfig.model_validate(template_data)
        manifest_json = json.dumps(template_data, sort_keys=True, separators=(",", ":"))
        sweep_id = create_sweep_record(nuubot, template_config, manifest_json)
    else:
        sweep_id = SWEEP_ID

    sweep_record, sweeprun_records = load_sweep_records(nuubot, sweep_id)
finally:
    nuubot.stop()

print(f"ready sweep_id={sweep_record.sweep_id} sweepruns={len(sweeprun_records)}")

In [ ]:
# Configs
# Tiny notebook-local wrappers over each loaded sweeprun record.

class SweeprunConfig:
    def __init__(self, record: SweeprunRow) -> None:
        self.record = record
        self.sweeprun_id = record.sweeprun_id
        self.sweep_id = record.sweep_id
        self.symbol = BotrunConfig.model_validate(json.loads(record.config_json)).market.symbol


class BotConfig:
    def __init__(self, record: SweeprunRow) -> None:
        self.record = record
        self.config = BotrunConfig.model_validate(json.loads(record.config_json))


sweeprun_record = sweeprun_records[0]
sweeprun_config = SweeprunConfig(sweeprun_record)
bot_config = BotConfig(sweeprun_record)

In [5]:
# Data
# Load this sweeprun's bars and yield point-in-time snapshots.

class Data:
    def __init__(self, sweeprun_config: SweeprunConfig, bot_config: BotConfig) -> None:
        self.sweeprun_config = sweeprun_config
        self.config = bot_config.config
        self.interval = self.config.market.interval
        self.bars: list[Bar] = []
        self.load_ms = 0

    def init(self) -> None:
        started = time.perf_counter()
        self.bars = load_binance_bars(self.config)
        self.load_ms = int((time.perf_counter() - started) * 1000)

    def __iter__(self):
        for bar in self.bars:
            yield MarketSnapshot(bars={self.interval: bar})

    def results(self) -> dict:
        return {"load_ms": self.load_ms, "loaded_bars": len(self.bars)}


In [6]:
# Signaler Under Test
# EMA cross signaler. init(data) precomputes indicators and signals once.

class EmaCross:
    def __init__(self, bot_config: BotConfig) -> None:
        config = bot_config.config
        signaler_config = config.signalers[0]
        self.interval = signaler_config.interval
        self.fast = int(signaler_config.params["fast"])
        self.slow = int(signaler_config.params["slow"])
        self.partial = signaler_config.partial
        if self.fast <= 0 or self.slow <= 0:
            raise ValueError("EMA periods must be positive")
        if self.fast >= self.slow:
            raise ValueError("fast EMA must be lower than slow EMA")
        self.rows: dict[int, dict] = {}
        self.load_ms = 0
        self.warmup_bars = max(self.fast, self.slow) + 10

    def init(self, data: Data) -> None:
        start_ms = date_ms(data.config.backtest.start)
        warmup_count = sum(1 for bar in data.bars if bar.ts_ms < start_ms)
        if warmup_count < self.warmup_bars:
            raise RuntimeError(f"not enough warmup bars: need={self.warmup_bars} got={warmup_count}")

        started = time.perf_counter()
        fast_ema = None
        slow_ema = None
        previous_diff = None
        count = 0
        for bar in data.bars:
            signal = Signal()
            if bar.closed or self.partial:
                count += 1
                fast_ema = ema(fast_ema, bar.close, self.fast)
                slow_ema = ema(slow_ema, bar.close, self.slow)
                if count >= self.slow:
                    diff = fast_ema - slow_ema
                    if previous_diff is not None:
                        if previous_diff <= 0 < diff:
                            signal = Signal(entry=True, reason="ema_cross_up")
                        elif previous_diff >= 0 > diff:
                            signal = Signal(exit=True, reason="ema_cross_down")
                    previous_diff = diff
            self.rows[bar.ts_ms] = {"signal": signal, "ema_fast": fast_ema, "ema_slow": slow_ema}
        self.load_ms = int((time.perf_counter() - started) * 1000)
        self.warmup_count = warmup_count

    async def next(self, snapshot: MarketSnapshot) -> Signal:
        bar = snapshot.bars[self.interval]
        return self.rows[bar.ts_ms]["signal"]

    def values(self, ts_ms: int) -> dict:
        return self.rows[ts_ms]

    def results(self) -> dict:
        return {"signal_load_ms": self.load_ms, "warmup_bars": self.warmup_count}


def ema(previous: float | None, price: float, period: int) -> float:
    if previous is None:
        return price
    alpha = 2 / (period + 1)
    return price * alpha + previous * (1 - alpha)


In [7]:
# Risk Under Test
# Placeholder risk profile. Kept as a class because risk rules will grow here.

class Risk:
    def __init__(self, bot_config: BotConfig) -> None:
        self.config = bot_config.config

    async def init(self) -> None:
        pass

    async def next(self, snapshot: MarketSnapshot, signal: Signal, accounts: dict[str, "RuntimeAccount"]) -> Signal:
        return signal

    def results(self) -> dict:
        return {}


In [8]:
# Executor Under Test
# Full GHBot sweep executor ported from nuutrader6 hcsweeps_old.

@dataclass(frozen=True)
class Tick:
    ts_ms: int
    price: float
    leg: str
    index: int


def synthesize_ticks(candle: Bar, interval_ms: int = 60_000) -> list[Tick]:
    bullish = candle.close >= candle.open
    p1, l1, p2, l2 = (
        (candle.low, "low", candle.high, "high")
        if bullish
        else (candle.high, "high", candle.low, "low")
    )
    step = max(interval_ms // 3, 1)
    close_offset = max(interval_ms - 1000, 1)
    return [
        Tick(candle.ts_ms, candle.open, "open", 0),
        Tick(candle.ts_ms + step, p1, l1, 1),
        Tick(candle.ts_ms + 2 * step, p2, l2, 2),
        Tick(candle.ts_ms + close_offset, candle.close, "close", 3),
    ]


@dataclass
class GhBotParams:
    initial_capital: float = 1000.0
    grid_capital_usd: float = 500.0
    hedge_capital_usd: float = 500.0
    grid_bound_pct: float = 5.0
    upper_bound_pct: float | None = None
    lower_bound_pct: float | None = None
    grid_levels: int = 41
    grid_spread_multiplier: float = 1.0
    hedge_trigger_pct: float = 1.0
    hedge_sl_pct: float = 0.5
    hedge_tsl_pct: float = 0.5
    hedge_stop_mode: str = "sl_then_tsl"
    hedge_max_losses: int = 0
    hedge_cooldown_hours: float = 4.0
    min_notional_usd: float = 10.0
    slippage_reserve_pct: float = 1.0
    comm_rate: float = 0.05
    taker_fee_pct: float = 0.0005
    slippage_pct: float = 0.0001


@dataclass
class GridLevel:
    index: int
    entry_price: float
    exit_price: float
    units: float
    is_boundary: bool
    state: str


@dataclass
class Hedge:
    entry_price: float
    units: float
    stop_price: float
    trail_extreme: float


@dataclass
class GridHedgeSweepBot:
    params: GhBotParams
    entry_price: float = 0.0
    lower_bound: float = 0.0
    upper_bound: float = 0.0
    hedge_trigger: float = 0.0
    levels: list[GridLevel] = field(default_factory=list)
    basis_by_level: dict[int, tuple[float, float]] = field(default_factory=dict)
    inventory_units: float = 0.0
    grid_realized: float = 0.0
    hedge_realized: float = 0.0
    fees: float = 0.0
    fills: list[dict] = field(default_factory=list)
    prev_price: float = 0.0
    hedge: Hedge | None = None
    hedge_cooldown_until_ms: int | None = None
    hedge_losses: int = 0
    phase: str = "grid_running"
    done: bool = False
    exit_reason: str | None = None
    hedge_cycles: int = 0
    ticks_processed: int = 0

    def on_entry(self, ts_ms: int, price: float) -> None:
        self.__init__(self.params)
        self.entry_price = price
        self.prev_price = price
        self.lower_bound = price * (1.0 - gh_lower_pct(self.params) / 100.0)
        self.upper_bound = price * (1.0 + gh_upper_pct(self.params) / 100.0)
        self.hedge_trigger = price * (1.0 - self.params.hedge_trigger_pct / 100.0)
        self.levels = build_levels(price, self.params)
        self.preload_above_start(ts_ms)

    def on_candle(self, candle: Bar) -> None:
        if self.done:
            return
        for tick in synthesize_ticks(candle):
            self.on_price(tick.ts_ms, tick.price)
            self.ticks_processed += 1
            if self.done:
                return

    def on_price(self, ts_ms: int, price: float) -> None:
        prev = self.prev_price
        if self.phase == "grid_running":
            self.process_grid_segment(ts_ms, prev, price)
        self.process_hedge(ts_ms, price)
        self.should_exit(ts_ms, price)
        self.prev_price = price

    def finish(self, final_ts_ms: int, final_price: float, end_reason: str) -> dict:
        if not self.done:
            self.close_grid_inventory(final_ts_ms, final_price, end_reason)
            if self.hedge is not None:
                self.close_hedge(final_ts_ms, final_price, False)
            self.done = True
            self.phase = "done"
            self.exit_reason = end_reason
        pnl_gross = self.grid_realized + self.hedge_realized
        last = self.fills[-1] if self.fills else None
        return {
            "fills": list(self.fills),
            "pnl_gross": pnl_gross,
            "fees": self.fees,
            "pnl_net": pnl_gross - self.fees,
            "exit_reason": self.exit_reason or end_reason,
            "exit_ts_ms": last["ts_ms"] if last else final_ts_ms,
            "exit_price": last["price"] if last else final_price,
            "hedge_cycles": self.hedge_cycles,
            "ticks_processed": self.ticks_processed,
        }

    def fill_price(self, price: float, side: str) -> float:
        return price * (1.0 + (1.0 if side == "buy" else -1.0) * self.params.slippage_pct)

    def preload_above_start(self, ts_ms: int) -> None:
        spacing = (self.upper_bound - self.lower_bound) / (self.params.grid_levels - 1)
        half_gap = spacing / 2.0
        above = [
            (idx, level)
            for idx, level in enumerate(self.levels)
            if not level.is_boundary and level.entry_price > self.entry_price
        ]
        nearest_above = min(above, key=lambda item: item[1].entry_price)[0] if above else None
        for idx, level in enumerate(self.levels):
            if level.is_boundary or level.entry_price <= self.entry_price:
                continue
            if idx == nearest_above and abs(level.entry_price - self.entry_price) < half_gap:
                continue
            entry_px = self.fill_price(self.entry_price, "buy")
            units = level.units
            if units * entry_px < self.params.min_notional_usd:
                continue
            fee = entry_px * units * self.params.taker_fee_pct
            self.fees += fee
            self.inventory_units += units
            self.basis_by_level[level.index] = (entry_px, units)
            level.state = "resting_exit"
            self.fills.append(gh_fill(ts_ms, "grid_preload", level.index, "buy", entry_px, units, fee))

    def process_grid_segment(self, ts_ms: int, prev: float, curr: float) -> None:
        hits = []
        for idx, level in enumerate(self.levels):
            if level.is_boundary:
                continue
            if level.state == "resting_entry" and curr <= prev and curr <= level.entry_price <= prev:
                hits.append((idx, level.entry_price, "buy"))
            elif level.state == "resting_exit" and curr >= prev and prev <= level.exit_price <= curr:
                hits.append((idx, level.exit_price, "sell"))
        hits.sort(key=lambda item: item[1], reverse=curr < prev)
        for idx, trigger, side in hits:
            self.fill_grid_level(ts_ms, idx, trigger, side)

    def fill_grid_level(self, ts_ms: int, idx: int, limit: float, side: str) -> None:
        level = self.levels[idx]
        if level.state == "closed":
            return
        fill_px = self.fill_price(limit, side)
        units = level.units
        if units * fill_px < self.params.min_notional_usd:
            return
        fee = fill_px * units * self.params.taker_fee_pct
        self.fees += fee
        if side == "buy":
            self.inventory_units += units
            self.basis_by_level[level.index] = (fill_px, units)
            level.state = "resting_exit"
        else:
            self.inventory_units -= units
            basis = self.basis_by_level.pop(level.index, None)
            if basis is not None:
                self.grid_realized += (fill_px - basis[0]) * basis[1]
            level.state = "resting_entry"
        self.fills.append(gh_fill(ts_ms, "grid", level.index, side, fill_px, units, fee))

    def close_grid_inventory(self, ts_ms: int, price: float, leg: str) -> None:
        if self.inventory_units <= 0.0 or not self.basis_by_level:
            return
        close_px = self.fill_price(price, "sell")
        open_levels = list(self.basis_by_level.items())
        self.basis_by_level.clear()
        for level_index, (basis, units) in open_levels:
            fee = close_px * units * self.params.taker_fee_pct
            self.fees += fee
            self.grid_realized += (close_px - basis) * units
            for level in self.levels:
                if level.index == level_index:
                    level.state = "resting_entry"
                    break
            self.fills.append(gh_fill(ts_ms, leg, level_index, "sell", close_px, units, fee))
        self.inventory_units = 0.0

    def process_hedge(self, ts_ms: int, price: float) -> None:
        if self.hedge is not None:
            use_tsl = self.params.hedge_stop_mode == "tsl" or self.phase == "hedge_only_tsl"
            if use_tsl and price < self.hedge.trail_extreme:
                self.hedge.trail_extreme = price
                self.hedge.stop_price = price * (1.0 + self.params.hedge_tsl_pct / 100.0)
            stop_price = self.hedge.stop_price
            if price >= stop_price:
                terminal_tsl = self.phase == "hedge_only_tsl"
                self.close_hedge(ts_ms, stop_price, not terminal_tsl)
                if terminal_tsl:
                    self.done = True
                    self.phase = "done"
                    self.exit_reason = "hedge_tsl"
            return
        if (
            self.phase != "grid_running"
            or self.inventory_units <= 0.0
            or not self.cooldown_elapsed(ts_ms)
        ):
            return
        if price <= self.hedge_trigger:
            self.open_hedge(ts_ms, self.hedge_trigger)

    def open_hedge(self, ts_ms: int, trigger_px: float) -> None:
        entry_px = self.fill_price(trigger_px, "sell")
        notional = min(self.inventory_units * entry_px, self.params.hedge_capital_usd)
        if notional < self.params.min_notional_usd:
            return
        units = notional / entry_px
        if self.params.hedge_stop_mode == "tsl":
            stop_price = entry_px * (1.0 + self.params.hedge_tsl_pct / 100.0)
        else:
            stop_price = entry_px * (1.0 + self.params.hedge_sl_pct / 100.0)
        fee = entry_px * units * self.params.taker_fee_pct
        self.fees += fee
        self.hedge = Hedge(entry_px, units, stop_price, entry_px)
        self.hedge_cycles += 1
        self.fills.append(gh_fill(ts_ms, "hedge_open", None, "sell", entry_px, units, fee))

    def close_hedge(self, ts_ms: int, price: float, set_cooldown: bool) -> None:
        if self.hedge is None:
            return
        hedge = self.hedge
        self.hedge = None
        close_px = self.fill_price(price, "buy")
        fee = close_px * hedge.units * self.params.taker_fee_pct
        self.fees += fee
        self.hedge_realized += (hedge.entry_price - close_px) * hedge.units
        if set_cooldown:
            self.hedge_losses += 1
            self.hedge_cooldown_until_ms = ts_ms + int(self.params.hedge_cooldown_hours * 3_600_000.0)
        self.fills.append(gh_fill(ts_ms, "hedge_close", None, "buy", close_px, hedge.units, fee))

    def should_exit(self, ts_ms: int, price: float) -> bool:
        if self.params.hedge_max_losses > 0 and self.hedge_losses >= self.params.hedge_max_losses:
            self.close_grid_inventory(ts_ms, price, "hedge_max_losses")
            self.done = True
            self.phase = "done"
            self.exit_reason = "hedge_max_losses"
            return True
        if self.phase != "grid_running":
            return False
        if price >= self.upper_bound:
            self.close_grid_inventory(ts_ms, price, "bound_favorable")
            if self.hedge is not None:
                self.close_hedge(ts_ms, price, False)
            self.done = True
            self.phase = "done"
            self.exit_reason = "bound_favorable"
            return True
        if price <= self.lower_bound:
            self.close_grid_inventory(ts_ms, price, "bound_adverse")
            if self.hedge is not None and self.params.hedge_stop_mode != "sl":
                self.hedge.trail_extreme = min(price, self.hedge.trail_extreme)
                self.hedge.stop_price = self.hedge.trail_extreme * (1.0 + self.params.hedge_tsl_pct / 100.0)
                self.phase = "hedge_only_tsl"
            else:
                if self.hedge is not None:
                    self.close_hedge(ts_ms, price, False)
                self.done = True
                self.phase = "done"
                self.exit_reason = "bound_adverse_no_hedge"
            return self.done
        return False

    def cooldown_elapsed(self, ts_ms: int) -> bool:
        return self.hedge_cooldown_until_ms is None or ts_ms >= self.hedge_cooldown_until_ms


def run_ghbot_executor(manifest: dict, candles: list[Bar], signals: list[dict], entry_signal: dict) -> dict:
    side = "long" if entry_signal["signal_kind"] == "long_enter" else "short"
    params = gh_params(manifest["bot"].get("params", {}))
    run_candles, final_ts_ms, final_price, end_reason = ghbot_run_window(candles, signals, entry_signal, side)
    if side == "short":
        entry_price = run_candles[0].open
        mirrored = [mirror_candle(candle, entry_price) for candle in run_candles]
        run = run_ghbot_until(params, mirrored, final_ts_ms, mirror_price(final_price, entry_price), end_reason)
        run["fills"] = [mirror_fill(fill, entry_price, params.taker_fee_pct) for fill in run["fills"]]
        recompute_run_amounts(run)
    else:
        run = run_ghbot_until(params, run_candles, final_ts_ms, final_price, end_reason)
    transactions = ghbot_transactions(run["fills"])
    fills = [fill for tx in transactions for fill in tx["fills"]]
    if not fills:
        raise ValueError("GHBot produced no fills")
    return {
        "executor_kind": "ghbot",
        "side": side,
        "entry_ts_ms": fills[0]["ts_ms"],
        "exit_ts_ms": run["exit_ts_ms"],
        "entry_price": fills[0]["price"],
        "exit_price": run["exit_price"],
        "pnl_gross": run["pnl_gross"],
        "fees": run["fees"],
        "pnl_net": run["pnl_net"],
        "exit_reason": run["exit_reason"],
        "ticks_processed": run["ticks_processed"],
        "config_json": manifest["bot"].get("params", {}),
        "metrics_bot_json": {
            "hedge_cycles": run["hedge_cycles"],
            "ticks": run["ticks_processed"],
            "fills": len(fills),
            "transactions": len(transactions),
            "side_mode": side,
        },
        "transactions": transactions,
    }


def run_ghbot_until(params: GhBotParams, candles: list[Bar], final_ts_ms: int, final_price: float, end_reason: str) -> dict:
    bot = GridHedgeSweepBot(params)
    bot.on_entry(candles[0].ts_ms, candles[0].open)
    for candle in candles:
        bot.on_candle(candle)
        if bot.done:
            break
    return bot.finish(final_ts_ms, final_price, end_reason)


def ghbot_run_window(candles: list[Bar], signals: list[dict], entry_signal: dict, side: str) -> tuple[list[Bar], int, float, str]:
    opposite_kind = "short_enter" if side == "long" else "long_enter"
    opposite = min(
        (signal for signal in signals if signal["ts_ms"] > entry_signal["ts_ms"] and signal["signal_kind"] == opposite_kind),
        key=lambda signal: signal["ts_ms"],
        default=None,
    )
    if opposite is not None:
        exit_index = next((i for i, candle in enumerate(candles) if candle.ts_ms >= opposite["ts_ms"]), None)
        if exit_index is not None and exit_index > 0:
            return candles[:exit_index], opposite["ts_ms"], opposite["price"], "signal_opposite_exit"
    last = candles[-1]
    return candles, last.ts_ms + 60_000, last.close, "force_close_end_data"
def build_levels(entry_price: float, params: GhBotParams) -> list[GridLevel]:
    lower = entry_price * (1.0 - gh_lower_pct(params) / 100.0)
    upper = entry_price * (1.0 + gh_upper_pct(params) / 100.0)
    spacing = (upper - lower) / (params.grid_levels - 1)
    active_count = params.grid_levels - 2
    usable_capital = params.grid_capital_usd * (1.0 - params.slippage_reserve_pct / 100.0)
    per_level_notional = usable_capital / active_count
    units = per_level_notional / (entry_price * (1.0 + params.comm_rate / 100.0))
    levels = []
    for idx in range(params.grid_levels):
        entry = lower + spacing * idx
        is_boundary = idx == 0 or idx + 1 == params.grid_levels
        levels.append(
            GridLevel(
                index=idx,
                entry_price=entry,
                exit_price=entry + spacing * params.grid_spread_multiplier,
                units=0.0 if is_boundary else units,
                is_boundary=is_boundary,
                state="closed" if is_boundary else "resting_entry",
            )
        )
    return levels


def ghbot_transactions(fills: list[dict]) -> list[dict]:
    groups: list[dict] = []
    grid_groups: dict[int, int] = {}
    active_hedge: int | None = None
    for fill in fills:
        converted = dict(fill)
        converted["metrics_json"] = {}
        leg = fill["leg"]
        if leg in {
            "grid_preload",
            "grid",
            "bound_favorable",
            "bound_adverse",
            "force_close_end_data",
            "signal_opposite_exit",
            "hedge_max_losses",
        }:
            grid_level = fill["grid_level"]
            if grid_level is None:
                raise ValueError("grid fill missing grid_level")
            group_index = grid_groups.get(grid_level)
            if group_index is None:
                groups.append({"transaction_kind": "grid", "leg": leg, "grid_level": grid_level, "fills": [], "metrics_json": {}})
                group_index = len(groups) - 1
                grid_groups[grid_level] = group_index
            groups[group_index]["fills"].append(converted)
        elif leg == "hedge_open":
            if active_hedge is not None:
                raise ValueError("overlapping hedge transaction")
            groups.append({"transaction_kind": "hedge", "leg": leg, "grid_level": None, "fills": [converted], "metrics_json": {}})
            active_hedge = len(groups) - 1
        elif leg == "hedge_close":
            if active_hedge is None:
                raise ValueError("hedge close without open")
            groups[active_hedge]["fills"].append(converted)
            active_hedge = None
        else:
            raise ValueError(f"unsupported GHBot fill leg {leg!r}")
    if active_hedge is not None:
        raise ValueError("hedge transaction ended open")
    return groups


def gh_params(params: dict) -> GhBotParams:
    return GhBotParams(**{key: value for key, value in params.items() if key in GhBotParams.__dataclass_fields__})


def gh_upper_pct(params: GhBotParams) -> float:
    return params.grid_bound_pct if params.upper_bound_pct is None else float(params.upper_bound_pct)


def gh_lower_pct(params: GhBotParams) -> float:
    return params.grid_bound_pct if params.lower_bound_pct is None else float(params.lower_bound_pct)


def gh_fill(ts_ms: int, leg: str, grid_level: int | None, side: str, price: float, size: float, fee: float) -> dict:
    return {"ts_ms": ts_ms, "leg": leg, "grid_level": grid_level, "side": side, "price": price, "size": size, "fee": fee}


def mirror_price(price: float, entry_price: float) -> float:
    return max(2.0 * entry_price - price, 2.2250738585072014e-308)


def mirror_candle(candle: Bar, entry_price: float) -> Bar:
    return Bar(
        ts_ms=candle.ts_ms,
        open=mirror_price(candle.open, entry_price),
        high=mirror_price(candle.low, entry_price),
        low=mirror_price(candle.high, entry_price),
        close=mirror_price(candle.close, entry_price),
        volume=candle.volume,
    )


def mirror_fill(fill: dict, entry_price: float, fee_rate: float) -> dict:
    price = mirror_price(fill["price"], entry_price)
    out = dict(fill)
    out["price"] = price
    out["fee"] = price * fill["size"] * fee_rate
    out["side"] = "sell" if fill["side"] == "buy" else "buy"
    return out


def recompute_run_amounts(run: dict) -> None:
    run["fees"] = sum(fill["fee"] for fill in run["fills"])
    run["pnl_gross"] = sum((-fill["price"] * fill["size"]) if fill["side"] == "buy" else (fill["price"] * fill["size"]) for fill in run["fills"])
    run["pnl_net"] = run["pnl_gross"] - run["fees"]
    if run["fills"]:
        run["exit_ts_ms"] = run["fills"][-1]["ts_ms"]
        run["exit_price"] = run["fills"][-1]["price"]




@dataclass
class RuntimeAccount:
    role: str
    name: str
    recon_count: int = 0
    fills: int = 0
    fees_usd: float = 0.0
    cash_usd: float = 0.0

    async def recon(self, snapshot: MarketSnapshot) -> None:
        self.recon_count += 1

    def record_fill(self, fill: dict) -> None:
        notional = float(fill["price"]) * float(fill["size"])
        fee = float(fill["fee"])
        self.fills += 1
        self.fees_usd += fee
        self.cash_usd += notional - fee if fill["side"] == "sell" else -notional - fee

    def summary(self) -> dict:
        return {
            "name": self.name,
            "recon_count": self.recon_count,
            "fills": self.fills,
            "fees_usd": round(self.fees_usd, 6),
            "cash_usd": round(self.cash_usd, 6),
        }


@dataclass
class GhbotResult:
    pnl_pct: float
    fees_pct: float
    wins: int
    losses: int
    trades: int
    cycles: int
    hedge_cycles: int
    grid_fills: int


def ghbot_params_from_config(config) -> GhBotParams:
    grid = config.grid
    hedge = config.hedge
    risk = config.risk
    return GhBotParams(
        initial_capital=float(grid.grid_investment_usdc) + (float(hedge.hedge_investment_usdc) if hedge.hedge_enabled else 0.0),
        grid_capital_usd=float(grid.grid_investment_usdc),
        hedge_capital_usd=float(hedge.hedge_investment_usdc) if hedge.hedge_enabled else 0.0,
        upper_bound_pct=float(config.upper_bound_pct),
        lower_bound_pct=float(config.lower_bound_pct),
        grid_levels=int(grid.grid_levels),
        grid_spread_multiplier=float(grid.grid_spread_multiplier),
        hedge_trigger_pct=float(hedge.hedge_entry_pct),
        hedge_sl_pct=float(hedge.hedge_sl_pct),
        hedge_tsl_pct=float(hedge.hedge_trailing_stop_pct),
        hedge_stop_mode="sl_then_tsl",
        hedge_max_losses=int(risk.risk_max_hedge_losses),
        hedge_cooldown_hours=float(hedge.hedge_cooldown_s) / 3600.0,
        min_notional_usd=1.0,
        slippage_reserve_pct=float(grid.grid_slippage_reserve_pct),
        taker_fee_pct=FEE_PCT / 100.0,
        slippage_pct=0.0,
    )


def ghbot_fill_role(fill: dict) -> str:
    return "hedge" if str(fill["leg"]).startswith("hedge") else "grid"


class Ghbot:
    def __init__(self, bot_config: BotConfig) -> None:
        config = bot_config.config.executor
        self.params = ghbot_params_from_config(config)
        self.grid_account = config.grid.grid_account
        self.hedge_account = config.hedge.hedge_account
        self.max_cycles = config.max_cycles
        self.bot: GridHedgeSweepBot | None = None
        self.fill_index = 0
        self.pnl_pct = 0.0
        self.fees_pct = 0.0
        self.wins = 0
        self.losses = 0
        self.trades = 0
        self.cycles = 0
        self.hedge_cycles = 0
        self.grid_fills = 0
        self.events: list[dict] = []
        self.equity: list[tuple[int, float]] = []

    async def init(self) -> dict[str, RuntimeAccount]:
        return {
            "grid": RuntimeAccount(role="grid", name=self.grid_account),
            "hedge": RuntimeAccount(role="hedge", name=self.hedge_account),
        }

    async def next(self, snapshot: MarketSnapshot, risk: Signal, accounts: dict[str, RuntimeAccount]) -> None:
        bar = snapshot.bars[next(iter(snapshot.bars))]
        if self.bot is not None and risk.exit:
            self._finish_cycle(bar.ts_ms, bar.close, "signal_opposite_exit", accounts)

        if self.bot is None and risk.entry and self._can_enter():
            self.bot = GridHedgeSweepBot(self.params)
            self.bot.on_entry(bar.ts_ms, bar.open)
            self.trades += 1
            self.events.append({"event": "ghbot_start", "ts_ms": bar.ts_ms, "price": bar.open, "reason": risk.reason})
            self._sync_fills(accounts)

        if self.bot is not None:
            self.bot.on_candle(bar)
            self._sync_fills(accounts)
            if self.bot.done:
                self._finish_cycle(bar.ts_ms, bar.close, self.bot.exit_reason or "done", accounts)

        self.equity.append((bar.ts_ms, self.pnl_pct + self._open_pnl_pct(bar.close)))

    async def stop(self, bar: Bar | None) -> None:
        if self.bot is not None and bar is not None:
            self._finish_cycle(bar.ts_ms, bar.close, "force_close_end_data", {})

    def result(self) -> GhbotResult:
        return GhbotResult(self.pnl_pct, self.fees_pct, self.wins, self.losses, self.trades, self.cycles, self.hedge_cycles, self.grid_fills)

    def _finish_cycle(self, ts_ms: int, price: float, reason: str, accounts: dict[str, RuntimeAccount]) -> None:
        if self.bot is None:
            return
        run = self.bot.finish(ts_ms, price, reason)
        self._sync_fills(accounts)
        cycle_pct = run["pnl_net"] / self.params.initial_capital * 100.0
        self.pnl_pct += cycle_pct
        self.fees_pct += run["fees"] / self.params.initial_capital * 100.0
        self.wins += int(cycle_pct >= 0)
        self.losses += int(cycle_pct < 0)
        self.cycles += 1
        self.hedge_cycles += int(run["hedge_cycles"])
        self.grid_fills += sum(1 for fill in run["fills"] if ghbot_fill_role(fill) == "grid")
        self.events.append({
            "event": "ghbot_stop",
            "ts_ms": run["exit_ts_ms"],
            "price": run["exit_price"],
            "reason": run["exit_reason"],
            "pnl_pct": cycle_pct,
            "ticks_processed": run["ticks_processed"],
        })
        self.bot = None
        self.fill_index = 0

    def _sync_fills(self, accounts: dict[str, RuntimeAccount]) -> None:
        if self.bot is None:
            return
        for fill in self.bot.fills[self.fill_index:]:
            role = ghbot_fill_role(fill)
            account = accounts.get(role)
            if account is not None:
                account.record_fill(fill)
            event = dict(fill)
            event["event"] = fill["leg"]
            event["account"] = account.name if account is not None else role
            self.events.append(event)
        self.fill_index = len(self.bot.fills)

    def _open_pnl_pct(self, price: float) -> float:
        if self.bot is None:
            return 0.0
        grid_open = sum((price - basis) * units for basis, units in self.bot.basis_by_level.values())
        hedge_open = 0.0 if self.bot.hedge is None else (self.bot.hedge.entry_price - price) * self.bot.hedge.units
        total = self.bot.grid_realized + self.bot.hedge_realized + grid_open + hedge_open - self.bot.fees
        return total / self.params.initial_capital * 100.0

    def _can_enter(self) -> bool:
        return self.max_cycles == 0 or self.cycles < self.max_cycles


In [9]:
# SweepRuntime
# Orchestrates one sweeprun: init components, loop snapshots, call component next(), collect results.

class SweepRuntime:
    def __init__(self, sweeprun_config: SweeprunConfig, bot_config: BotConfig, data: Data, signaler: EmaCross, risk: Risk, executor: Ghbot) -> None:
        self.sweeprun_config = sweeprun_config
        self.bot_config = bot_config
        self.config = bot_config.config
        self.data = data
        self.signaler = signaler
        self.risk = risk
        self.executor = executor
        self.accounts: dict[str, RuntimeAccount] = {}
        self.start_ms = date_ms(self.config.backtest.start)
        self.stop_ms = date_ms(self.config.backtest.stop)
        self.last_bar = None
        self.chart_rows = []
        self.signal_count = 0
        self.signal_ms = 0
        self.execution_ms = 0
        self.loop_ms = 0
        self.elapsed_ms = 0

    async def init(self) -> None:
        started = time.perf_counter()
        self.data.init()
        self.signaler.init(self.data)
        await self.risk.init()
        self.accounts = await self.executor.init()
        self.elapsed_ms = int((time.perf_counter() - started) * 1000)

    async def run(self) -> None:
        started = time.perf_counter()
        for snapshot in self.data:
            bar = snapshot.bars[self.config.market.interval]
            if bar.ts_ms > self.stop_ms:
                break
            await self.next(snapshot)
        self.loop_ms = int((time.perf_counter() - started) * 1000)
        await self.executor.stop(self.last_bar)

    async def next(self, snapshot: MarketSnapshot) -> None:
        bar = snapshot.bars[self.config.market.interval]
        signal_started = time.perf_counter()
        signal = await self.signaler.next(snapshot)
        self.signal_ms += int((time.perf_counter() - signal_started) * 1000)
        if bar.ts_ms < self.start_ms:
            return

        for account in self.accounts.values():
            await account.recon(snapshot)
        risk_signal = await self.risk.next(snapshot, signal, self.accounts)
        if risk_signal.entry or risk_signal.exit:
            self.signal_count += 1

        execution_started = time.perf_counter()
        await self.executor.next(snapshot, risk_signal, self.accounts)
        self.execution_ms += int((time.perf_counter() - execution_started) * 1000)
        self.last_bar = bar
        self.chart_rows.append({
            "ts_ms": bar.ts_ms,
            "open": bar.open,
            "high": bar.high,
            "low": bar.low,
            "close": bar.close,
            "ema_fast": self.signaler.values(bar.ts_ms)["ema_fast"],
            "ema_slow": self.signaler.values(bar.ts_ms)["ema_slow"],
            "signal": "entry" if risk_signal.entry else "exit" if risk_signal.exit else "",
        })

    def results(self) -> dict:
        result = self.executor.result()
        return {
            "result": result.__dict__,
            "events": self.executor.events,
            "equity": self.executor.equity,
            "chart_rows": self.chart_rows,
            "bars": len(self.chart_rows),
            "warmup_bars": self.signaler.results()["warmup_bars"],
            "signals": self.signal_count,
            "accounts": {role: account.summary() for role, account in self.accounts.items()},
            "timing": {
                "elapsed_ms": self.elapsed_ms + self.loop_ms,
                "load_ms": self.data.results()["load_ms"],
                "signal_load_ms": self.signaler.results()["signal_load_ms"],
                "core_loop_ms": self.loop_ms,
                "signal_ms": self.signal_ms,
                "execution_ms": self.execution_ms,
            },
        }


In [10]:
# Run SweepRun
# Run every sweeprun in this sweep and keep all outputs in memory.

sweep_outputs = []
for row in sweeprun_records:
    sweeprun_config = SweeprunConfig(row)
    bot_config = BotConfig(row)
    data = Data(sweeprun_config, bot_config)
    signaler = EmaCross(bot_config)
    risk = Risk(bot_config)
    executor = Ghbot(bot_config)
    runtime = SweepRuntime(sweeprun_config, bot_config, data, signaler, risk, executor)

    await runtime.init()
    await runtime.run()
    output = runtime.results()
    output["sweeprun_id"] = row.sweeprun_id
    output["sweeprun_index"] = row.sweeprun_index
    sweep_outputs.append(output)

assert sweep_outputs
assert all(output["bars"] > 0 for output in sweep_outputs)
assert all(output["result"]["cycles"] <= BotConfig(row).config.executor.max_cycles or BotConfig(row).config.executor.max_cycles == 0 for output, row in zip(sweep_outputs, sweeprun_records, strict=True))

sweep_output = sweep_outputs[0]
print(f"sweep_id={sweep_record.sweep_id} sweepruns={len(sweep_outputs)} bars={sweep_output['bars']} first_cycles={sweep_output['result']['cycles']}")


sweep_id=25 sweepruns=1 bars=129600 first_cycles=1848


In [ ]:
# Save Results
# Persist sweeprun timing and summary back to the sweeps database.

nuubot = Nuubot().setup()
try:
    by_id = {output["sweeprun_id"]: output for output in sweep_outputs}
    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        for row in sweeprun_records:
            output = by_id[row.sweeprun_id]
            loaded = session.get(SweeprunRow, row.sweeprun_id)
            loaded.status = "complete"
            loaded.results_json = json.dumps({"result": output["result"], "timing": output["timing"]}, sort_keys=True)
        sweep = session.get(SweepRow, sweep_record.sweep_id)
        sweep.status = "complete"
        sweep.results_json = json.dumps({
            "sweepruns": len(sweep_outputs),
            "first_result": sweep_outputs[0]["result"],
            "timing": {"elapsed_ms": sum(output["timing"]["elapsed_ms"] for output in sweep_outputs)},
        }, sort_keys=True)
        session.commit()
finally:
    nuubot.stop()

print("results saved")

In [12]:
# Raw Results
# JSON/debug output for investigation without dumping every trade event.

def iso(ts_ms: int) -> str:
    return datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).isoformat()


def row_for(output: dict) -> dict:
    result = output["result"]
    first_ts = output["chart_rows"][0]["ts_ms"] if output["chart_rows"] else 0
    last_ts = output["chart_rows"][-1]["ts_ms"] if output["chart_rows"] else 0
    return {
        "sweeprun_id": output["sweeprun_id"],
        "sweeprun_index": output["sweeprun_index"],
        "start": iso(first_ts),
        "end": iso(last_ts),
        "duration_days": round((last_ts - first_ts) / 86_400_000, 2) if first_ts and last_ts else 0,
        "bots": 1,
        "symbols": 1,
        "bars": output["bars"],
        "warmup_bars": output["warmup_bars"],
        "signals": output["signals"],
        "pnl_pct": round(result["pnl_pct"], 6),
        "fees_pct": round(result["fees_pct"], 6),
        "trades": result["trades"],
        "wins": result["wins"],
        "losses": result["losses"],
        "win_rate_pct": round(result["wins"] / result["cycles"] * 100, 2) if result["cycles"] else 0,
        "run_ms": output["timing"]["core_loop_ms"],
        "signal_load_ms": output["timing"]["signal_load_ms"],
        "signal_ms": output["timing"]["signal_ms"],
        "load_ms": output["timing"]["load_ms"],
        "total_ms": output["timing"]["elapsed_ms"],
    }


def result_rows() -> list[dict]:
    return [row_for(output) for output in sweep_outputs]


rows = result_rows()
print(json.dumps({
    "rows": rows,
    "event_count": len(sweep_output["events"]),
    "events_preview": sweep_output["events"][:20],
    "result": sweep_output["result"],
    "accounts": sweep_output["accounts"],
}, indent=2))


{
  "rows": [
    {
      "sweeprun_id": 25,
      "sweeprun_index": 0,
      "start": "2025-01-01T00:00:00+00:00",
      "end": "2025-03-31T23:59:00+00:00",
      "duration_days": 90.0,
      "bots": 1,
      "symbols": 1,
      "bars": 129600,
      "warmup_bars": 44640,
      "signals": 3696,
      "pnl_pct": 17.615505,
      "fees_pct": 0.0,
      "trades": 1848,
      "wins": 672,
      "losses": 1176,
      "win_rate_pct": 36.36,
      "run_ms": 1291,
      "signal_load_ms": 184,
      "signal_ms": 0,
      "load_ms": 1786,
      "total_ms": 3265
    }
  ],
  "event_count": 81184,
  "events_preview": [
    {
      "event": "ghbot_start",
      "ts_ms": 1735690500000,
      "price": 93656.19,
      "reason": "ema_cross_up"
    },
    {
      "ts_ms": 1735690500000,
      "leg": "grid_preload",
      "grid_level": 21,
      "side": "buy",
      "price": 93656.19,
      "size": 0.0005418099843674067,
      "fee": 0.0,
      "event": "grid_preload",
      "account": "sgrid"
    },
  

In [13]:
# Report
# Final scroll-to-bottom chart and concise result panel.


chart_rows = sweep_output["chart_rows"]
if not chart_rows:
    print("no chart rows")
else:
    rows = result_rows()
    report = rows[0]
    display_rows = chart_rows[-5000:]
    categories = [iso(row["ts_ms"])[:16].replace("T", " ") for row in display_rows]
    candles = [[row["open"], row["close"], row["low"], row["high"]] for row in display_rows]
    ema_fast = [row["ema_fast"] for row in display_rows]
    ema_slow = [row["ema_slow"] for row in display_rows]
    marks = [
        {"coord": [i, row["close"]], "value": row["signal"], "itemStyle": {"color": "#facc15" if row["signal"] == "entry" else "#38bdf8"}}
        for i, row in enumerate(display_rows)
        if row["signal"]
    ]

    def field(label: str, value) -> str:
        return f"<div class='field'><span>{label}</span><b>{value}</b></div>"

    html = f"""
    <div id='emacross-ghbot-report'>
      <div id='emacross-ghbot-chart' style='height:520px;width:100%;'></div>
      <div class='summary'>
        <section>
          <h3>Runtime Stats</h3>
          {field('Period Start', report['start'])}
          {field('Period End', report['end'])}
          {field('Duration', str(report['duration_days']) + ' days')}
          {field('Symbols', report['symbols'])}
          {field('Bars', report['bars'])}
          {field('Warmup Bars', report['warmup_bars'])}
          {field('Chart Points', len(display_rows))}
        </section>
        <section>
          <h3>Performance</h3>
          {field('# Bots', report['bots'])}
          {field('Trades', report['trades'])}
          {field('Win Rate', str(report['win_rate_pct']) + '%')}
          {field('PnL', str(report['pnl_pct']) + '%')}
          {field('Fees', str(report['fees_pct']) + '%')}
        </section>
        <section>
          <h3>SweepRun Stats</h3>
          {field('Time Taken', str(report['run_ms']) + ' ms')}
          {field('Signal Load', str(report['signal_load_ms']) + ' ms')}
        </section>
      </div>
    </div>
    <style>
      #emacross-ghbot-report {{ font-family: system-ui, Segoe UI, sans-serif; color: #e5e7eb; background: #111827; padding: 14px; }}
      #emacross-ghbot-report .summary {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-top: 12px; }}
      #emacross-ghbot-report section {{ border: 1px solid #374151; padding: 12px; background: #0f172a; }}
      #emacross-ghbot-report h3 {{ margin: 0 0 10px; font-size: 15px; }}
      #emacross-ghbot-report .field {{ display: flex; justify-content: space-between; gap: 12px; padding: 5px 0; border-top: 1px solid #1f2937; font-size: 13px; }}
      #emacross-ghbot-report .field span {{ color: #9ca3af; }}
      #emacross-ghbot-report .field b {{ color: #f9fafb; font-weight: 600; text-align: right; }}
    </style>
    <script src='https://cdn.jsdelivr.net/npm/echarts@5/dist/echarts.min.js'></script>
    <script>
      (() => {{
        const chart = echarts.init(document.getElementById('emacross-ghbot-chart'));
        chart.setOption({{
          animation: false,
          backgroundColor: '#111827',
          tooltip: {{ trigger: 'axis' }},
          legend: {{ data: ['Candles', 'EMA Fast', 'EMA Slow'], textStyle: {{ color: '#e5e7eb' }} }},
          grid: {{ left: 60, right: 30, top: 50, bottom: 80 }},
          xAxis: {{ type: 'category', data: {json.dumps(categories)}, axisLabel: {{ color: '#9ca3af' }} }},
          yAxis: {{ scale: true, axisLabel: {{ color: '#9ca3af' }}, splitLine: {{ lineStyle: {{ color: '#1f2937' }} }} }},
          dataZoom: [{{ type: 'inside' }}, {{ type: 'slider', bottom: 20 }}],
          series: [
            {{ name: 'Candles', type: 'candlestick', data: {json.dumps(candles)}, itemStyle: {{ color: '#22c55e', color0: '#ef4444', borderColor: '#22c55e', borderColor0: '#ef4444' }}, markPoint: {{ data: {json.dumps(marks)} }} }},
            {{ name: 'EMA Fast', type: 'line', data: {json.dumps(ema_fast)}, smooth: false, showSymbol: false, lineStyle: {{ color: '#facc15', width: 1.5 }} }},
            {{ name: 'EMA Slow', type: 'line', data: {json.dumps(ema_slow)}, smooth: false, showSymbol: false, lineStyle: {{ color: '#38bdf8', width: 1.5 }} }}
          ]
        }});
      }})();
    </script>
    """
    report_path = Path("workspace/results/emacross-ghbot-report.html")
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(html, encoding="utf-8")
    display(IFrame(str(report_path), width="100%", height=900))
